# Quantale Training Submission
## Sumeya Ibrahim
## Domain: API Rate Limit Tiers

This notebook implements a Residuated Quantale for modeling API rate limit delegation in microservice architectures.

**Domain**: Cloud API gateway rate limiting tiers
- `blocked`: No requests allowed (0 req/s)
- `limited`: Minimal access (10 req/s)
- `standard`: Normal access (100 req/s)
- `elevated`: High throughput (1000 req/s)
- `unlimited`: No rate limit (top tier)

**⊗ (meet/min)**: Combining two rate limits gives the MORE RESTRICTIVE one. This models the intersection of policies: if service A allows standard and service B allows limited, the effective limit is limited.

**Unit = unlimited**: The top element acts as identity since min(unlimited, x) = x.

**Right residual (→)**: Given a service with rate limit `a` and a downstream budget cap `c`, what is the maximum rate limit the service can safely delegate to a child process without exceeding `c`?

In [1]:
from __future__ import annotations
from typing import TypeVar, Generic, FrozenSet, Callable, Optional, List
from itertools import product as cartesian

T = TypeVar("T")

---
## Stage 1 — FiniteSet
Define your universe Q. At least 4 elements.

In [2]:
class FiniteSet(Generic[T]):
    """A finite set: unordered, no duplicates, supports membership and subsets."""

    def __init__(self, elements: list[T]) -> None:
        seen = []
        for e in elements:
            if e not in seen:
                seen.append(e)
        self._elements: list[T] = seen

    def __contains__(self, x: object) -> bool:
        return x in self._elements

    def __iter__(self):
        return iter(self._elements)

    def __len__(self) -> int:
        return len(self._elements)

    def __repr__(self) -> str:
        return "{" + ", ".join(str(e) for e in self._elements) + "}"

    def is_subset_of(self, other: "FiniteSet[T]") -> bool:
        return all(e in other for e in self)

    def power_set(self) -> list[FrozenSet[T]]:
        elems = list(self._elements)
        result = []
        for mask in range(1 << len(elems)):
            result.append(frozenset(elems[i] for i in range(len(elems)) if mask & (1 << i)))
        return result

    def cartesian_product(self) -> list[tuple[T, T]]:
        return list(cartesian(self._elements, repeat=2))

    def check_no_duplicates(self) -> bool:
        return len(self._elements) == len(set(str(e) for e in self._elements))

    def to_list(self) -> list[T]:
        return self._elements.copy()

### YOUR DOMAIN DEFINITION — Stage 1
Define your set Q below. Replace with your own elements.

In [3]:
# STUDENT CONFIGURATION — EDIT ONLY THIS BLOCK

MY_ELEMENTS = ["blocked", "limited", "standard", "elevated", "unlimited"]

# Define the direct cover edges of YOUR partial order
MY_DIRECT_EDGES = [
    ("blocked", "limited"),
    ("limited", "standard"),
    ("standard", "elevated"),
    ("elevated", "unlimited"),
]

# Define your monoid multiplication function
def MY_MUL(a: str, b: str) -> str:
    order = {"blocked": 0, "limited": 1, "standard": 2, "elevated": 3, "unlimited": 4}
    idx_a = order[a]
    idx_b = order[b]
    result_idx = min(idx_a, idx_b)
    reverse_order = {v: k for k, v in order.items()}
    return reverse_order[result_idx]

MY_UNIT = "unlimited"

# END STUDENT CONFIGURATION

In [4]:
# VERIFIER: Identity block — checks your elements are different from training
TRAINING_ELEMENTS = {"none", "read", "append", "write", "admin"}
MY_SET = set(MY_ELEMENTS)

assert MY_SET != TRAINING_ELEMENTS, "ERROR: Your domain must be different from the training example!"

assert len(MY_ELEMENTS) >= 4, f"ERROR: You need at least 4 elements. You have {len(MY_ELEMENTS)}."

Q = FiniteSet(MY_ELEMENTS)
print(f"Stage 1 PASSED: Q = {Q}")
print(f"  |Q| = {len(Q)} elements")
print(f"  |Q×Q| = {len(Q.cartesian_product())} ordered pairs")
print(f"  |P(Q)| = {len(Q.power_set())} subsets")

Stage 1 PASSED: Q = {blocked, limited, standard, elevated, unlimited}
  |Q| = 5 elements
  |Q×Q| = 25 ordered pairs
  |P(Q)| = 32 subsets


---
## Stage 2 — BinaryRelation
A binary relation R ⊆ Q×Q.

In [5]:
class BinaryRelation(Generic[T]):
    """A binary relation R ⊆ Q×Q."""

    def __init__(self, base: FiniteSet[T], pairs: list[tuple[T, T]]) -> None:
        self.base = base
        for a, b in pairs:
            assert a in base and b in base, f"Pair ({a},{b}) outside base set"
        self._pairs: FrozenSet[tuple[T, T]] = frozenset(pairs)

    def __contains__(self, pair: tuple[T, T]) -> bool:
        return pair in self._pairs

    def holds(self, a: T, b: T) -> bool:
        return (a, b) in self._pairs

    def is_reflexive(self) -> bool:
        return all(self.holds(a, a) for a in self.base)

    def is_symmetric(self) -> bool:
        return all(self.holds(b, a) for (a, b) in self._pairs)

    def is_antisymmetric(self) -> bool:
        for (a, b) in self._pairs:
            if a != b and self.holds(b, a):
                return False
        return True

    def is_transitive(self) -> bool:
        for (a, b) in self._pairs:
            for c in self.base:
                if self.holds(b, c) and not self.holds(a, c):
                    return False
        return True

    def closure(self) -> "BinaryRelation[T]":
        elems = list(self.base)
        reach = {(a, b): self.holds(a, b) for a in elems for b in elems}
        for e in elems:
            reach[(e, e)] = True
        for k in elems:
            for a in elems:
                for b in elems:
                    if reach[(a, k)] and reach[(k, b)]:
                        reach[(a, b)] = True
        pairs = [(a, b) for (a, b), v in reach.items() if v]
        return BinaryRelation(self.base, pairs)

    def __repr__(self) -> str:
        pairs = sorted(str(p) for p in self._pairs)
        return f"Relation({', '.join(pairs)})"

### YOUR DOMAIN — Stage 2 & 3
Define your direct edges. The notebook computes the reflexive-transitive closure automatically.

In [6]:
direct = BinaryRelation(Q, MY_DIRECT_EDGES)
leq = direct.closure()

print(f"Direct edges: {MY_DIRECT_EDGES}")
print(f"Reflexive-transitive closure has {len(leq._pairs)} pairs")

Direct edges: [('blocked', 'limited'), ('limited', 'standard'), ('standard', 'elevated'), ('elevated', 'unlimited')]
Reflexive-transitive closure has 15 pairs


---
## Stage 3 — Poset
Partially ordered set (Q, ≤).

In [7]:
class Poset(Generic[T]):
    """Partially ordered set (Q, ≤). ≤ must be reflexive, antisymmetric, transitive."""

    def __init__(self, base: FiniteSet[T], leq: BinaryRelation[T]) -> None:
        self.base = base
        self.leq = leq
        self._validate()

    def _validate(self) -> None:
        assert self.leq.is_reflexive(),     "≤ must be reflexive"
        assert self.leq.is_antisymmetric(), "≤ must be antisymmetric"
        assert self.leq.is_transitive(),    "≤ must be transitive"

    def le(self, a: T, b: T) -> bool:
        return self.leq.holds(a, b)

    def lt(self, a: T, b: T) -> bool:
        return self.le(a, b) and a != b

    def comparable(self, a: T, b: T) -> bool:
        return self.le(a, b) or self.le(b, a)

    def upper_bounds(self, subset: list[T]) -> list[T]:
        return [c for c in self.base if all(self.le(x, c) for x in subset)]

    def lower_bounds(self, subset: list[T]) -> list[T]:
        return [c for c in self.base if all(self.le(c, x) for x in subset)]

    def hasse_edges(self) -> list[tuple[T, T]]:
        edges = []
        for a in self.base:
            for b in self.base:
                if self.lt(a, b):
                    between = [c for c in self.base
                               if c != a and c != b
                               and self.lt(a, c) and self.lt(c, b)]
                    if not between:
                        edges.append((a, b))
        return edges

    def __repr__(self) -> str:
        edges = self.hasse_edges()
        return f"Poset({self.base}, covers={edges})"

In [8]:
P = Poset(Q, leq)

print("Stage 3 PASSED: Poset is valid")
print("  Reflexive:    ✓")
print("  Antisymmetric: ✓")
print("  Transitive:   ✓")
print(f"Hasse cover edges: {P.hasse_edges()}")
print("Order queries:")
for a, b in [("blocked", "unlimited"), ("limited", "standard"), ("elevated", "standard")]:
    sym = "≤" if P.le(a, b) else "≰"
    print(f"  {a} {sym} {b}")

Stage 3 PASSED: Poset is valid
  Reflexive:    ✓
  Antisymmetric: ✓
  Transitive:   ✓
Hasse cover edges: [('blocked', 'limited'), ('limited', 'standard'), ('standard', 'elevated'), ('elevated', 'unlimited')]
Order queries:
  blocked ≤ unlimited
  limited ≤ standard
  elevated ≰ standard


---
## Stage 4 — Lattice
Every pair has a join (⋁) and meet (⋀).

In [9]:
class Lattice(Poset[T]):
    """Lattice: poset where every pair has join (⋁) and meet (⋀)."""

    def __init__(self, base: FiniteSet[T], leq: BinaryRelation[T]) -> None:
        super().__init__(base, leq)
        self._validate_lattice()

    def _validate_lattice(self) -> None:
        for a in self.base:
            for b in self.base:
                assert self._find_join(a, b) is not None, f"No join for ({a}, {b})"
                assert self._find_meet(a, b) is not None, f"No meet for ({a}, {b})"

    def _find_join(self, a: T, b: T) -> Optional[T]:
        ubs = self.upper_bounds([a, b])
        candidates = [c for c in ubs if all(self.le(c, d) for d in ubs)]
        return candidates[0] if candidates else None

    def _find_meet(self, a: T, b: T) -> Optional[T]:
        lbs = self.lower_bounds([a, b])
        candidates = [c for c in lbs if all(self.le(d, c) for d in lbs)]
        return candidates[0] if candidates else None

    def join(self, a: T, b: T) -> T:
        result = self._find_join(a, b)
        assert result is not None
        return result

    def meet(self, a: T, b: T) -> T:
        result = self._find_meet(a, b)
        assert result is not None
        return result

In [10]:
L = Lattice(Q, leq)

print("Stage 4 PASSED: Lattice is valid")
print("  Every pair has join and meet")
print("Join/Meet examples:")
examples = [
    ("limited", "standard"),
    ("standard", "elevated"),
    ("blocked", "unlimited"),
]
for a, b in examples:
    print(f"  {a} ⋁ {b} = {L.join(a, b)},   {a} ⋀ {b} = {L.meet(a, b)}")

Stage 4 PASSED: Lattice is valid
  Every pair has join and meet
Join/Meet examples:
  limited ⋁ standard = standard,   limited ⋀ standard = limited
  standard ⋁ elevated = elevated,   standard ⋀ elevated = standard
  blocked ⋁ unlimited = unlimited,   blocked ⋀ unlimited = blocked


---
## Stage 5 — CompleteLattice
Join and meet for ANY subset. ⊤ and ⊥ defined.

In [11]:
class CompleteLattice(Lattice[T]):
    """Complete lattice: join/meet for ANY subset. ⊤ = ⋁Q, ⊥ = ⋀Q."""

    @property
    def top(self) -> T:
        return self.big_join(list(self.base))

    @property
    def bottom(self) -> T:
        return self.big_meet(list(self.base))

    def big_join(self, subset: list[T]) -> T:
        if not subset:
            return self.big_meet(list(self.base))
        result = subset[0]
        for x in subset[1:]:
            result = self.join(result, x)
        return result

    def big_meet(self, subset: list[T]) -> T:
        if not subset:
            candidates = [c for c in self.base
                        if all(self.le(x, c) for x in self.base)]
            assert candidates
            return candidates[0]
        result = subset[0]
        for x in subset[1:]:
            result = self.meet(result, x)
        return result

In [12]:
CL = CompleteLattice(Q, leq)

print(f"Stage 5 PASSED: Complete Lattice")
print(f"  ⊤ (top)    = {CL.top}")
print(f"  ⊥ (bottom) = {CL.bottom}")
print("Arbitrary joins:")
subsets = [
    ["limited", "standard", "elevated"],
    ["blocked", "limited"],
    ["elevated", "unlimited"],
]
for s in subsets:
    print(f"  ⋁{s} = {CL.big_join(s)},   ⋀{s} = {CL.big_meet(s)}")

Stage 5 PASSED: Complete Lattice
  ⊤ (top)    = unlimited
  ⊥ (bottom) = blocked
Arbitrary joins:
  ⋁['limited', 'standard', 'elevated'] = elevated,   ⋀['limited', 'standard', 'elevated'] = limited
  ⋁['blocked', 'limited'] = limited,   ⋀['blocked', 'limited'] = blocked
  ⋁['elevated', 'unlimited'] = unlimited,   ⋀['elevated', 'unlimited'] = elevated


---
## Stage 6 — Monoid
Associative binary operation ⊗ with identity element e.

In [13]:
class MonoidMixin(Generic[T]):
    """Mixin adding monoid operation ⊗ with identity element e."""

    def __init__(self, mul_fn: Callable[[T, T], T], unit: T) -> None:
        self._mul_fn = mul_fn
        self._unit = unit

    def mul(self, a: T, b: T) -> T:
        return self._mul_fn(a, b)

    @property
    def unit(self) -> T:
        return self._unit

    def check_monoid(self, base: FiniteSet[T]) -> dict:
        elems = list(base)
        result = {"closure": True, "associativity": True, "identity": True,
                  "counterexamples": []}

        for a in elems:
            for b in elems:
                if self.mul(a, b) not in base:
                    result["closure"] = False
                    result["counterexamples"].append(
                        f"closure: {a}⊗{b}={self.mul(a,b)} ∉ Q")

        for a in elems:
            for b in elems:
                for c in elems:
                    lhs = self.mul(self.mul(a, b), c)
                    rhs = self.mul(a, self.mul(b, c))
                    if lhs != rhs:
                        result["associativity"] = False
                        result["counterexamples"].append(
                            f"assoc: ({a}⊗{b})⊗{c}={lhs} ≠ {a}⊗({b}⊗{c})={rhs}")

        for a in elems:
            if self.mul(self._unit, a) != a or self.mul(a, self._unit) != a:
                result["identity"] = False
                result["counterexamples"].append(
                    f"identity: unit⊗{a}={self.mul(self._unit, a)}")

        return result

In [14]:
M = MonoidMixin(MY_MUL, MY_UNIT)
report = M.check_monoid(Q)

print(f"Stage 6 PASSED: Monoid is valid")
print(f"  Closure:        {report['closure']}")
print(f"  Associativity:  {report['associativity']}")
print(f"  Identity:       {report['identity']} (unit={MY_UNIT})")
print("Composition examples:")
examples = [
    ("elevated", "limited"),
    ("standard", "elevated"),
    ("unlimited", "standard"),
    ("blocked", "unlimited"),
]
for a, b in examples:
    print(f"  {a} ⊗ {b} = {M.mul(a, b)}")

Stage 6 PASSED: Monoid is valid
  Closure:        True
  Associativity:  True
  Identity:       True (unit=unlimited)
Composition examples:
  elevated ⊗ limited = limited
  standard ⊗ elevated = standard
  unlimited ⊗ standard = standard
  blocked ⊗ unlimited = blocked


---
## Stage 7 — Quantale
Complete lattice + monoid where ⊗ distributes over ⋁.

In [15]:
class Quantale(CompleteLattice[T], MonoidMixin[T]):
    """Quantale: complete lattice + monoid where ⊗ distributes over ⋁."""

    def __init__(
        self,
        base: FiniteSet[T],
        leq: BinaryRelation[T],
        mul_fn: Callable[[T, T], T],
        unit: T,
    ) -> None:
        CompleteLattice.__init__(self, base, leq)
        MonoidMixin.__init__(self, mul_fn, unit)
        self._validate_quantale()

    def _validate_quantale(self) -> None:
        elems = list(self.base)
        for a in elems:
            for b in elems:
                for c in elems:
                    join_bc = self.join(b, c)
                    lhs = self.mul(a, join_bc)
                    rhs = self.join(self.mul(a, b), self.mul(a, c))
                    assert lhs == rhs, (
                        f"Left distributivity fails: {a}⊗({b}⋁{c})={lhs} ≠ ({a}⊗{b})⋁({a}⊗{c})={rhs}"
                    )
                    lhs2 = self.mul(join_bc, a)
                    rhs2 = self.join(self.mul(b, a), self.mul(c, a))
                    assert lhs2 == rhs2, (
                        f"Right distributivity fails: ({b}⋁{c})⊗{a}={lhs2} ≠ ({b}⊗{a})⋁({c}⊗{a})={rhs2}"
                    )

    def check_distributivity_report(self) -> dict:
        elems = list(self.base)
        failures = []
        for a in elems:
            for b in elems:
                for c in elems:
                    jbc = self.join(b, c)
                    if self.mul(a, jbc) != self.join(self.mul(a, b), self.mul(a, c)):
                        failures.append(f"left: {a}⊗({b}⋁{c})")
                    if self.mul(jbc, a) != self.join(self.mul(b, a), self.mul(c, a)):
                        failures.append(f"right: ({b}⋁{c})⊗{a}")
        return {"distributivity": len(failures) == 0, "failures": failures}

In [16]:
QTL = Quantale(Q, leq, MY_MUL, MY_UNIT)
dist = QTL.check_distributivity_report()

print(f"Stage 7 PASSED: Quantale distributivity holds")
print(f"  Distributivity: {dist['distributivity']}")
print(f"  Checked all {len(Q)**3} triples (a,b,c)")

Stage 7 PASSED: Quantale distributivity holds
  Distributivity: True
  Checked all 125 triples (a,b,c)


---
## Stage 8 — ResiduatedQuantale
Right/left residuals with Galois adjunction.

In [17]:
class ResiduatedQuantale(Quantale[T]):
    """Quantale with right/left residuals. Core adjunction: a⊗b≤c ⟺ b≤a→c."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self._rr_cache: dict[tuple, T] = {}
        self._lr_cache: dict[tuple, T] = {}

    def right_residual(self, a: T, c: T) -> T:
        key = (a, c)
        if key not in self._rr_cache:
            feasible = [b for b in self.base if self.le(self.mul(a, b), c)]
            self._rr_cache[key] = self.big_join(feasible) if feasible else self.bottom
        return self._rr_cache[key]

    def left_residual(self, c: T, b: T) -> T:
        key = (c, b)
        if key not in self._lr_cache:
            feasible = [a for a in self.base if self.le(self.mul(a, b), c)]
            self._lr_cache[key] = self.big_join(feasible) if feasible else self.bottom
        return self._lr_cache[key]

    def verify_adjunction(self) -> dict:
        failures = []
        elems = list(self.base)
        for a in elems:
            for b in elems:
                for c in elems:
                    lhs  = self.le(self.mul(a, b), c)
                    mid  = self.le(b, self.right_residual(a, c))
                    rhs  = self.le(a, self.left_residual(c, b))
                    if not (lhs == mid == rhs):
                        failures.append(
                            f"({a},{b},{c}): a⊗b≤c={lhs}, b≤a→c={mid}, a≤c←b={rhs}"
                        )
        return {"adjunction_holds": len(failures) == 0, "failures": failures[:5]}

    def can_do(self, role: T, required: T) -> bool:
        return self.le(role, required)

    def effective_permission(self, roles: list[T]) -> T:
        return self.big_join(roles)

    def max_delegatable(self, own_permission: T, cap: T) -> T:
        return self.right_residual(own_permission, cap)

    def compose(self, perm_a: T, perm_b: T) -> T:
        return self.mul(perm_a, perm_b)

In [18]:
RQ = ResiduatedQuantale(Q, leq, MY_MUL, MY_UNIT)
adj = RQ.verify_adjunction()

print(f"Stage 8 PASSED: Galois adjunction holds")
print(f"  Adjunction: {adj['adjunction_holds']}")
print(f"  Checked all {len(Q)**3} triples (a,b,c)")
print("Right residual examples (a → c):")
examples = [
    ("elevated", "standard", "max delegation: elevated service, budget=standard"),
    ("limited", "elevated", "max delegation: limited service, budget=elevated"),
    ("unlimited", "standard", "max delegation: unlimited service, budget=standard"),
]
for a, c, desc in examples:
    res = RQ.right_residual(a, c)
    print(f"  {a} → {c} = {res:10s}  ({desc})")

Stage 8 PASSED: Galois adjunction holds
  Adjunction: True
  Checked all 125 triples (a,b,c)
Right residual examples (a → c):
  elevated → standard = standard    (max delegation: elevated service, budget=standard)
  limited → elevated = unlimited   (max delegation: limited service, budget=elevated)
  unlimited → standard = standard    (max delegation: unlimited service, budget=standard)


---
## Domain Queries — YOUR 3 QUERIES

Write at least one `le`, one `big_join`, and one `right_residual` call.
Each must have a comment explaining what the result means in YOUR domain.

In [ ]:
# DOMAIN QUERY 1: le (partial order comparison)
# Query: Can a 'standard' tier service handle requests that require 'elevated'?
result1 = RQ.can_do("standard", "elevated")
print(f"Query 1 — can_do('standard', 'elevated'): {result1}")
print("INTERPRETATION: A 'standard' tier (100 req/s) CANNOT handle 'elevated' (1000 req/s) traffic.")
print("The result is False because standard ≤ elevated, meaning standard is MORE RESTRICTIVE.")

# DOMAIN QUERY 2: big_join (least upper bound / effective permission)
# Query: A user has API keys with 'limited' and 'standard' tiers.
result2 = RQ.effective_permission(["limited", "standard"])
print(f"Query 2 — effective_permission(['limited', 'standard']): {result2}")
print("INTERPRETATION: User gets the BEST available tier across all keys: standard (100 req/s).")

# DOMAIN QUERY 3: right_residual (max safe delegation)
# Query: Gateway has 'elevated' (1000 req/s), downstream budget is 'standard' (100 req/s).
result3 = RQ.max_delegatable("elevated", "standard")
print(f"Query 3 — max_delegatable('elevated', 'standard'): {result3}")
print("INTERPRETATION: Gateway can only delegate 'standard' tier to stay within budget.")
print("Verify: elevated ⊗ standard = min(1000, 100) = 100 ≤ standard ✓")

---
## Final Summary

All 8 stages verified. Your quantale is ready for the application repository.

In [ ]:
print("Quantale Submission Complete")
print(f"Domain: API Rate Limit Tiers")
print(f"Elements: blocked < limited < standard < elevated < unlimited")
print(f"⊗ (meet): min — combining tiers gives the more restrictive")
print(f"Unit: unlimited — identity element")
print(f"Residual: max safe delegation without exceeding budget cap")
print("All verifier cells passed. Ready for application repository.")